<a href="https://colab.research.google.com/github/Aditi-k1912/Gen-AI/blob/main/Pract4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

tf.keras.utils.set_random_seed(42)

text = [
    "how are you doing today",
    "how are you feeling now",
    "i am going to the market",
    "i am going to the office",
    "i will call you later",
    "i will call you tomorrow",
    "see you at the office tomorrow",
    "see you at the market later",
    "what are you doing today",
    "what time will you reach",
    "thank you so much for your help",
    "thank you for the update",
    "please send me the report",
    "please send me the details",
    "good morning how are you",
    "good night see you tomorrow",
    "are you coming to the office today",
    "are you coming to the market",
    "i am happy to help you",
    "let me know when you reach",
]

# Build vocabulary (0 is reserved for padding)
words = sorted(set(" ".join(text).split()))
word_index = {w: i + 1 for i, w in enumerate(words)}
index_word = {i: w for w, i in word_index.items()}
vocab_size = len(word_index) + 1

# Create input-sequence / next-word pairs
sequences = []
for line in text:
    tokens = [word_index[w] for w in line.split()]
    for i in range(1, len(tokens)):
        sequences.append(tokens[: i + 1])

max_len = max(len(s) for s in sequences)
padded = np.array([[0] * (max_len - len(s)) + s
                   for s in sequences])
X, y = padded[:, :-1], padded[:, -1]
print("Training sequences:", X.shape[0])

model = Sequential([
    Embedding(vocab_size, 32, mask_zero=True),
    LSTM(64),
    Dense(vocab_size, activation="softmax"),
])
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
history = model.fit(X, y, epochs=300, verbose=0)
print("Training Accuracy:",
      round(history.history["accuracy"][-1], 2))

def suggest(prefix, k=3):
    tokens = [word_index[w] for w in prefix.lower().split()
              if w in word_index][-(max_len - 1):]
    seq = np.array([[0] * (max_len - 1 - len(tokens)) + tokens])
    probs = model.predict(seq, verbose=0)[0]
    top = probs.argsort()[-k:][::-1]
    return [index_word[i] for i in top if i != 0]

for prefix in ["i am", "thank you", "see you at the",
               "please send me the"]:
    print(prefix, "->", suggest(prefix))


Training sequences: 91
Training Accuracy: 0.87
i am -> ['going', 'happy', 'to']
thank you -> ['for', 'so', 'at']
see you at the -> ['office', 'market', 'tomorrow']
please send me the -> ['report', 'details', 'office']
